# Hypotetiske oljeutslipp under ekstremværet «Ingunn» – OpenDrift/OpenOil

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/andreaszahl/oilspill/blob/main/notebooks/openoil_ingunn.ipynb)

Notebooken simulerer et **tenkt** utslipp fra hver produksjonsinnretning i Norskehavet (64–68° N)
under ekstremværet «Ingunn» (31. januar–1. februar 2024). Den bruker OpenOil, oljemodellen i
[OpenDrift](https://opendrift.github.io) fra Meteorologisk institutt.

**Ikke et beredskapsverktøy.** Utslippene er hypotetiske. Notebooken er laget til en emneoppgave
i Geografisk IT og KI (UiA).

**Slik bruker du den**
1. Kjør med `TESTMODUS = True` først (6 timer, 200 partikler, 2 innretninger).
2. Sett `TESTMODUS = False` og kjør alt på nytt (*Runtime → Run all*).
3. Siste celle laster ned en zip med GeoJSON og PNG. Pakk `spills/` ut i `data/processed/spills/` i repoet og commit.

Notebooken er selvstendig. Innretninger og oljetyper leses fra repoet på GitHub, og strøm og vind
hentes fra thredds.met.no via OPeNDAP. Du trenger ingen innlogging.

## 1. Installer OpenDrift
Hvis Colab ber deg starte runtime på nytt etter installasjonen, gjør det og kjør videre fra cellen under.

In [ ]:
%pip install -q opendrift bottleneck
import opendrift
print("OpenDrift-versjon:", opendrift.__version__)

## 2. Konstanter
Alle valg som påvirker resultatet står her. **Oljetypen** står ikke i koden. Den hentes per felt fra
[`config/oil_types.csv`](https://github.com/andreaszahl/oilspill/blob/main/config/oil_types.csv).

In [ ]:
from datetime import datetime, timedelta
from pathlib import Path

# --- Scenario: ekstremværet «Ingunn» (alle tider i UTC) ---
STARTTID = datetime(2024, 1, 31, 0, 0)   # 2024-01-31 00:00 UTC
VARIGHET = 72                            # timer som simuleres
OUTPUT = 1                               # timer mellom lagrede tidssteg
TIDSSTEG_MIN = 15                        # modellens interne tidssteg (minutter)
UTSLIPPSVARIGHET = 12                    # timer med kontinuerlig lekkasje
UTSLIPPSRATE_M3_PER_T = 500              # m³ olje per time -> 6 000 m³ totalt (hypotetisk)
ANTALL_PARTIKLER = 2000
# OLJETYPE: hentes per felt fra config/oil_types.csv (se OLJETYPER_URL).

# --- Testmodus: kjør denne først ---
TESTMODUS = True
TEST_VARIGHET = 6
TEST_PARTIKLER = 200
TEST_INNRETNINGER = ["norne_fpso", "aasta_hansteen_spar"]  # Aasta Hansteen tester domenekanten

# --- Etterbehandling ---
POLYGON_BUFFER_M = 1000   # radius rundt hver overflatepartikkel (m), se avsnitt 7
MIN_MASSEANDEL_FLAK = 0.01  # partikkel med i flaket bare hvis >= 1 % av oljemassen er igjen
FORENKLING_M = 50         # forenkling av polygonkanter (m) for mindre filer
CRS_METRISK = "EPSG:25833"  # UTM 33N – areal og buffer i meter
CRS_LAGRING = "EPSG:4326"   # WGS84 – eksportformat

# --- Datakilder ---
REPO_RAW = "https://raw.githubusercontent.com/andreaszahl/oilspill/main"
INNRETNINGER_URL = f"{REPO_RAW}/data/processed/installations.geojson"
OLJETYPER_URL = f"{REPO_RAW}/config/oil_types.csv"
THREDDS = "https://thredds.met.no/thredds/dodsC"
NORKYST_URL = THREDDS + "/fou-hi/norkyst800m-1h/NorKyst-800m_ZDEPTHS_his.an.{d:%Y%m%d}00.nc"
MEPS_URL = THREDDS + "/meps25epsarchive/{d:%Y/%m/%d}/meps_det_sfc_{d:%Y%m%d}T00Z.ncml"
WW3_URL = THREDDS + "/ww3_4km_archive_files/{d:%Y/%m/%d}/ww3_4km_{d:%Y%m%d}T00Z.nc"
STROM_KILDE = "NorKyst800 (met.no)"
VIND_KILDE = "MEPS 2,5 km (MetCoOp/met.no)"
BOLGE_KILDE = "WAVEWATCH III 4 km (met.no)"

if TESTMODUS:
    VARIGHET, ANTALL_PARTIKLER = TEST_VARIGHET, TEST_PARTIKLER
SLUTTID = STARTTID + timedelta(hours=VARIGHET)

SUFFIKS = "_TEST" if TESTMODUS else ""
UT_DIR = Path("resultater" + SUFFIKS)        # det som pakkes i zip
for p in (UT_DIR / "spills", UT_DIR / "plots"):
    p.mkdir(parents=True, exist_ok=True)

print(f"{'TESTMODUS' if TESTMODUS else 'FULL KJØRING'}: {STARTTID} -> {SLUTTID} UTC, "
      f"{ANTALL_PARTIKLER} partikler, {UTSLIPPSRATE_M3_PER_T} m³/t i {UTSLIPPSVARIGHET} t")

## 3. Innretninger og oljetyper fra repoet
Innretningene kommer fra Sokkeldirektoratet (NLOD 2.0) via `scripts/fetch_facilities.py`. Hver
innretning kobles til en oljetype gjennom feltet (`field`). Aasta Hansteen har ingen egen olje i
OpenDrifts liste, så der brukes en stedfortreder (se `note`).

In [ ]:
import geopandas as gpd
import pandas as pd

alle_innretninger = gpd.read_file(INNRETNINGER_URL)
if "slug" not in alle_innretninger.columns:
    raise RuntimeError("installations.geojson på GitHub mangler 'slug'/'field'. "
                       "Er steg 1 (fetch_facilities.py) committet og pushet til main?")

oljetyper = pd.read_csv(OLJETYPER_URL)
alle_innretninger = alle_innretninger.merge(oljetyper[["field", "oil_type", "note"]], on="field", how="left")
mangler = alle_innretninger[alle_innretninger["oil_type"].isna()]
if len(mangler):
    raise ValueError(f"Ingen oljetype i config/oil_types.csv for felt: {sorted(mangler['field'])}")

innretninger = alle_innretninger
if TESTMODUS:
    innretninger = alle_innretninger[alle_innretninger["slug"].isin(TEST_INNRETNINGER)]

display(alle_innretninger[["name", "field", "oil_type", "lat", "lon", "note"]])
print("Simuleres nå:", ", ".join(innretninger["name"]))

### Valider oljenavnene mot OpenDrift
Navnene i `config/oil_types.csv` må stå nøyaktig slik de står i oljelista til den OpenDrift-versjonen
som er installert. Står et navn ikke i lista, stopper notebooken og foreslår de nærmeste treffene.

In [ ]:
import difflib
from opendrift.models.openoil import OpenOil

_o = OpenOil(loglevel=50, weathering_model="noaa")
try:
    GYLDIGE_OLJER = set(_o.get_configspec("seed:oil_type")["seed:oil_type"]["enum"])
except Exception:
    GYLDIGE_OLJER = set(_o.oiltypes)
print(f"{len(GYLDIGE_OLJER)} oljetyper i OpenDrift {opendrift.__version__}")

feil = []
for navn in sorted(set(oljetyper["oil_type"])):
    if navn in GYLDIGE_OLJER:
        print("  OK   ", navn)
    else:
        forslag = difflib.get_close_matches(navn, list(GYLDIGE_OLJER), n=3, cutoff=0.5)
        feil.append(f"{navn!r} – nærmeste treff: {forslag}")
if feil:
    raise ValueError("Disse oljetypene finnes ikke i den installerte OpenDrift-versjonen:\n  "
                     + "\n  ".join(feil) + "\nRett config/oil_types.csv og push på nytt.")

## 4. Strøm, vind og bølger fra thredds.met.no

**Strøm: NorKyst800.** Dette er havmodellen til met.no for norskekysten, med 800 m oppløsning og
timesverdier. Arkivet har én fil per døgn (00–23 UTC). Filene åpnes som ett datasett med
`open_mfdataset`, slik at det ikke blir hull mellom kl. 23 og 00.

**Vind: MEPS 2,5 km** (MetCoOp, met.no/SMHI/FMI). NorKyst-filene inneholder også vind (`Uwind/Vwind`).
Det er MEPS-vinden som driver havmodellen, men den er interpolert til NorKyst-gitteret, og domenet
er mindre. Derfor fjerner vi vinden fra NorKyst-readeren og bruker MEPS direkte. Slik kommer vinden
garantert fra én, navngitt kilde.

**Bølger: WAVEWATCH III 4 km** (met.no). Bølgemodellen gir signifikant bølgehøyde (Hs), periode og
Stokes-drift, altså nettotransporten fra bølgebevegelsen. Hs avgjør hvor mye olje brytende bølger
blander ned. Uten bølgemodell måtte OpenOil anslå bølgene fra vinden. Røyktesten viste at
standardanslaget (`use_tabularised_stokes_drift`, 25 km fetch) ga Hs = 2,2 m ved 20 m/s vind, mot
~12 m for fullt utviklet sjø. Da ble nesten ingen olje blandet ned under Ingunn. WW3-fila inneholder
også vind; den fjernes, slik at MEPS er eneste vindkilde.

For MEPS og WW3 lager vi et *beste estimat*: fra hver 00Z-kjøring brukes de første 24 timene. Det gir
den ferskeste prognosen for hvert tidspunkt. Arkivene har ingen egen analyse-tidsserie. Vi henter
bare et utsnitt (62–70° N, 0–18° Ø) for å slippe å laste ned hele modellområdet.

In [ ]:
import numpy as np
import xarray as xr
from opendrift.readers import reader_netCDF_CF_generic

def dager(fra, til):
    """Datoer (00 UTC) fra og med fra.date() til og med til.date()."""
    return list(pd.date_range(fra.date(), til.date(), freq="D").to_pydatetime())

# Små romlige blokker (150 x 150 celler) gjør at bare området rundt partiklene lastes ned.
# Med én blokk per tidssteg hentet OpenDrift hele NorKyst-gitteret (16 dyp) hver gang:
# 205 s mot 13 s for 2 simulerte timer i røyktesten.
NORKYST_BLOKKER = {"time": 1, "X": 150, "Y": 150}

def apne_flere(urls):
    try:
        from opendrift.readers import open_dataset_opendrift
        return open_dataset_opendrift(urls, chunks=dict(NORKYST_BLOKKER))
    except ImportError:
        return xr.open_mfdataset(urls, chunks=dict(NORKYST_BLOKKER), data_vars="minimal",
                                 coords="minimal", compat="override")

# --- NorKyst800 (strøm) ---
norkyst_urls = [NORKYST_URL.format(d=d) for d in dager(STARTTID, SLUTTID)]
print("NorKyst800-filer:", *norkyst_urls, sep="\n  ")
ds_norkyst = apne_flere(norkyst_urls)
ds_norkyst = ds_norkyst.drop_vars([v for v in ("Uwind", "Vwind") if v in ds_norkyst.variables])

# Vertikal blanding beregnes fra vinden (se avsnitt 6), ikke fra NorKysts diffusivitet (AKs),
# fordi AKs er 0 i overflatelaget. OpenDrift leser likevel variabelen og godtar bare [0, 1] m²/s.
# Under Ingunn gir NorKyst opptil ~1,4 m²/s, og ved Kristin også negative verdier (ned til −3,3,
# trolig en artefakt fra interpolasjonen mellom dybdenivåene). Vi klipper derfor til [0, 1].
# I NorKyst heter den AKs, med standard_name 'salinity_vertical_diffusion_coefficient'.
DIFFUSIVITET = {"ocean_vertical_diffusivity", "salinity_vertical_diffusion_coefficient"}
for v in list(ds_norkyst.data_vars):
    if v == "AKs" or ds_norkyst[v].attrs.get("standard_name") in DIFFUSIVITET:
        print(f"Klipper {v} til [0, 1] m²/s")
        ds_norkyst[v] = ds_norkyst[v].clip(min=0.0, max=1.0).assign_attrs(ds_norkyst[v].attrs)
norkyst = reader_netCDF_CF_generic.Reader(ds_norkyst, name=STROM_KILDE)

# --- MEPS (vind) og WW3 (bølger): beste estimat fra 00Z-kjøringer ---
def beste_estimat(url_mal, kjoringer, behold, lat_omr=(62.0, 70.0), lon_omr=(0.0, 18.0)):
    deler, utsnitt = [], None
    for i, d in enumerate(kjoringer):
        url = url_mal.format(d=d)
        print("  ", url)
        ds = xr.open_dataset(url, chunks={"time": 1})
        ds = ds[[v for v in behold if v in ds.variables]]
        if utsnitt is None:  # samme gitter i alle kjøringer -> beregn utsnittet én gang
            ydim, xdim = ds["latitude"].dims
            la, lo = ds["latitude"].values, ds["longitude"].values
            jj, ii = np.where((la >= lat_omr[0]) & (la <= lat_omr[1]) & (lo >= lon_omr[0]) & (lo <= lon_omr[1]))
            utsnitt = {ydim: slice(jj.min(), jj.max() + 1), xdim: slice(ii.min(), ii.max() + 1)}
        ds = ds.isel(**utsnitt)
        neste = kjoringer[i + 1] if i + 1 < len(kjoringer) else SLUTTID + timedelta(hours=1)
        deler.append(ds.sel(time=slice(d, neste - timedelta(seconds=1))))
    return xr.concat(deler, dim="time", data_vars="minimal", coords="minimal", compat="override")

kjoringer = dager(STARTTID, SLUTTID - timedelta(seconds=1))
print("MEPS-kjøringer:")
ds_meps = beste_estimat(MEPS_URL, kjoringer,
                        ("x_wind_10m", "y_wind_10m", "projection_lambert", "latitude", "longitude"))
meps = reader_netCDF_CF_generic.Reader(ds_meps, name=VIND_KILDE)

print("WW3-kjøringer:")
ds_ww3 = beste_estimat(WW3_URL, kjoringer,  # hs = Hs, tp/t02 = perioder, uuss/vuss = Stokes-drift
                       ("hs", "tp", "t02", "uuss", "vuss", "rotated_pole", "latitude", "longitude"))
ww3 = reader_netCDF_CF_generic.Reader(ds_ww3, name=BOLGE_KILDE)

assert "x_wind" not in norkyst.variables, "NorKyst-readeren skal ikke levere vind"
assert "x_wind" not in ww3.variables, "WW3-readeren skal ikke levere vind"
for r in (norkyst, meps, ww3):
    print(f"\n{r.name}: {r.start_time} -> {r.end_time}, variabler: {sorted(r.variables)}")

### Dekningssjekk før simulering
Vi sjekker at alle innretningene ligger inne i alle tre modellområdene, og at tidsrommet dekkes.
Aasta Hansteen ligger langt ute, bare noen titalls km fra ytterkanten av NorKyst800.
Partikler som driver ut av NorKyst-domenet får strøm = 0 (fallback) og drives da bare av vind og
bølger. Hvor mange det gjelder, telles etter hver simulering.

In [ ]:
def sjekk_dekning(reader, punkter):
    x, y = reader.lonlat2xy(punkter["lon"].values, punkter["lat"].values)
    x, y = np.atleast_1d(x), np.atleast_1d(y)
    inne = (x >= reader.xmin) & (x <= reader.xmax) & (y >= reader.ymin) & (y <= reader.ymax)
    kant = np.minimum.reduce([x - reader.xmin, reader.xmax - x, y - reader.ymin, reader.ymax - y])
    # WW3 har rotert lengde/bredde-gitter (grader); de andre er projisert i meter.
    kant_km = kant * 111.2 if "longlat" in str(getattr(reader, "proj4", "")) else kant / 1000
    tid_ok = (reader.start_time <= STARTTID) and (reader.end_time >= SLUTTID)
    return pd.DataFrame({"kilde": reader.name, "innretning": punkter["name"].values,
                         "inne_i_domenet": inne, "km_til_kant": kant_km.round(0), "tid_ok": tid_ok})

dekning = pd.concat([sjekk_dekning(r, alle_innretninger) for r in (norkyst, meps, ww3)], ignore_index=True)
display(dekning)

problemer = dekning[~dekning["inne_i_domenet"] | ~dekning["tid_ok"]]
if len(problemer):
    raise RuntimeError(f"Mangler dekning:\n{problemer.to_string(index=False)}")
for _, r in dekning[dekning["km_til_kant"] < 100].iterrows():
    print(f"ADVARSEL: {r['innretning']} ligger bare {r['km_til_kant']:.0f} km fra kanten av {r['kilde']}. "
          "Andelen partikler utenfor domenet rapporteres etter simuleringen.")
print("Dekningssjekk OK: alle innretninger og hele tidsrommet er dekket.")

## 5. Landmaske og kystlinje
OpenDrift bruker en innebygd landmaske (`roaring-landmask`) laget fra **GSHHG i full oppløsning**.
Et raster på 0,0042° (ca. 460 m nord–sør og 180 m øst–vest her) gir raske oppslag. Punkter nær
land sjekkes eksakt mot GSHHG-polygonene. Kystlinjen er altså så god som GSHHG «full», som er
generalisert i skjærgården: små holmer og smale sund mangler. Partikler som treffer land, blir
**strandet** og ligger der resten av simuleringen (`general:coastline_action = 'stranding'`).

Kartverkets kystlinje (N50/N250) ville gitt en mer detaljert kyst, særlig i skjærgården på Helgeland og
i Vestfjorden. Det er ikke implementert her.

De samme GSHHG-polygonene brukes til å klippe flakpolygonene, slik at kartet og modellen ser
den samme kysten.

In [ ]:
from shapely import wkb, box, clip_by_rect

def hent_land(bbox=(0.0, 62.0, 20.0, 70.0)):
    """GSHHG full-oppløsning i utsnittet (lon0, lat0, lon1, lat1), i EPSG:25833."""
    try:
        import roaring_landmask
        prov = roaring_landmask.LandmaskProvider.Gshhg
        roaring_landmask.Shapes.new(prov)
        alle = wkb.loads(roaring_landmask.Shapes.wkb(prov)).geoms
        kilde = "GSHHG full via roaring-landmask (samme som modellen)"
    except Exception as e:
        print("roaring-landmask-polygoner utilgjengelige, bruker cartopy:", e)
        import cartopy.feature as cfeature
        alle = list(cfeature.GSHHSFeature(scale="f", levels=[1]).geometries())
        kilde = "GSHHG full via cartopy"
    ramme = box(*bbox)
    biter = [clip_by_rect(g, *bbox) for g in alle if g.intersects(ramme)]
    land = gpd.GeoSeries(biter, crs=CRS_LAGRING).to_crs(CRS_METRISK).buffer(0).union_all()
    print(f"Land: {kilde}, {len(biter)} polygoner")
    return land

LAND = hent_land()

## 6. Modelloppsett (OpenOil)
For hver innretning:
- **Utslipp:** kontinuerlig fra overflaten i `UTSLIPPSVARIGHET` timer, med `UTSLIPPSRATE_M3_PER_T`
  m³/t. Partiklene slippes jevnt fordelt i tid innenfor 50 m fra innretningen.
- **Drift:**
  - overflatestrøm fra NorKyst800
  - vindens direkte drag på oljen (OpenOils standard vinddriftfaktor, skrives ut under)
  - Stokes-drift fra bølgemodellen WAVEWATCH III
  - horisontal diffusjon på 10 m²/s, som representerer turbulens modellene ikke oppløser
- **Vertikalt:** brytende bølger blander olje ned i vannsøylen, i takt med Hs fra WW3. Mye av
  den kommer opp igjen.
  Turbulensen beregnes fra vinden (`windspeed_Large1994`), ikke fra NorKyst. NorKysts vertikale
  diffusivitet er 0 i overflatelaget, og med den kom oljen aldri under overflaten, selv i orkan
  (oppdaget i røyktesten).
- **Forvitring (NOAA/ADIOS):** fordampning, emulsjon (vannopptak) og naturlig dispergering. Hvor
  raskt det skjer, avhenger av oljetypen, så valget i `config/oil_types.csv` betyr mye.
- **Land:** stranding, se avsnitt 5.
- **Utenfor NorKyst:** strøm = 0 som fallback, i stedet for at partiklene deaktiveres.
- **Storm:** `drift:max_speed = 3` m/s, slik at datablokken rundt partiklene er stor nok.
  NorKysts diffusivitet klippes til [0, 1] m²/s, så OpenDrift ikke forkaster verdiene.

In [ ]:
def sett(o, nokkel, verdi):
    try:
        o.set_config(nokkel, verdi)
    except Exception as e:
        print(f"ADVARSEL: kunne ikke sette {nokkel} = {verdi!r}: {e}")

def lag_modell(rad):
    o = OpenOil(loglevel=30, weathering_model="noaa")
    o.add_reader([meps, ww3, norkyst])  # vind kun fra MEPS, bølger kun fra WW3, strøm fra NorKyst
    sett(o, "general:use_auto_landmask", True)
    # Bestemmer hvor stor datablokk som hentes rundt partiklene. Standard 1,3 m/s er for lavt
    # under Ingunn (strøm + 3 % vinddrift + Stokes-drift); 3 m/s gir buffer på 16 NorKyst-celler.
    sett(o, "drift:max_speed", 3)
    sett(o, "general:coastline_action", "stranding")
    sett(o, "environment:fallback:x_sea_water_velocity", 0)
    sett(o, "environment:fallback:y_sea_water_velocity", 0)
    sett(o, "environment:constant:horizontal_diffusivity", 10)
    sett(o, "drift:vertical_mixing", True)
    # NorKysts diffusivitet er 0 i overflaten, og da blandes oljen aldri ned. Vindbasert profil
    # (Large mfl. 1994) gir blanding som øker med vindstyrken, slik den skal i en storm.
    sett(o, "vertical_mixing:diffusivitymodel", "windspeed_Large1994")
    sett(o, "drift:use_tabularised_stokes_drift", False)  # Stokes-drift og Hs fra WW3
    sett(o, "processes:evaporation", True)
    sett(o, "processes:emulsification", True)
    sett(o, "processes:dispersion", True)
    o.seed_elements(lon=float(rad["lon"]), lat=float(rad["lat"]), z=0, radius=50,
                    number=ANTALL_PARTIKLER,
                    time=[STARTTID, STARTTID + timedelta(hours=UTSLIPPSVARIGHET)],
                    m3_per_hour=UTSLIPPSRATE_M3_PER_T, oil_type=rad["oil_type"])
    return o

def vis_oppsett(o):
    for k in ("vertical_mixing:diffusivitymodel", "drift:max_speed", "seed:wind_drift_factor", "drift:wind_drift_depth", "environment:constant:horizontal_diffusivity",
              "drift:use_tabularised_stokes_drift", "general:coastline_action", "seed:oil_type"):
        try:
            print(f"  {k} = {o.get_config(k)}")
        except Exception:
            pass

## 7. Etterbehandling: partikler → polygoner og massebalanse
Etterbehandlingen gjøres i **EPSG:25833** (UTM 33N, meter). Buffer og areal krever meter, og
UTM 33N har lite arealforvrengning langs kysten her. Eksporten skjer i **EPSG:4326**.

**Polygon per time: buffer + union + negativ buffer («closing»)**
1. Hver partikkel *i overflaten* som fortsatt har minst `MIN_MASSEANDEL_FLAK` (1 %) av sin
   opprinnelige oljemasse, får en sirkel med radius r = `POLYGON_BUFFER_M`. Uten dette filteret ble
   partikler som har mistet nesten all olje til fordampning og dispergering, med i flaket. For
   Kristin ga det 532 km² flak med 0,0 % olje i overflaten.
2. Sirklene slås sammen (union). Partikler som ligger nærmere hverandre enn 2r, smelter sammen.
3. Polygonet krympes med r/2. Det glatter kantene og fjerner tynne «broer», mens flekker som har
   skilt lag forblir egne polygoner. Hver partikkel representerer da i praksis en skive med radius r/2.
4. Polygonet klippes mot land (GSHHG) og forenkles med `FORENKLING_M`.

Hvorfor ikke konkavt hull? Det lager ett omriss rundt alle punktene og bygger lett bro over øyer,
sund og åpent hav mellom adskilte flekker. Resultatet avhenger også mye av parameteren `ratio`.
Buffer-metoden er mer ærlig for et oppsplittet flak. **Ulempen** er at arealet avhenger av r, så
arealet er et mål på utbredelse, ikke på areal dekket av olje.

**Massebalanse** (andel av utsluppet masse ved hver time):
- `pct_surface`: olje i overflaten
- `pct_submerged`: midlertidig nedblandet olje
- `pct_evaporated`: fordampet
- `pct_dispersed`: permanent dispergert
- `pct_stranded`: strandet

Strandede partikler forsvinner fra OpenDrifts aktive liste, så verdiene deres «fryses» fra
strandingstidspunktet.

In [ ]:
import shapely

def ffill(a):
    """Fremoverfyll langs tid (akse 1), slik at deaktiverte partikler beholder siste verdi."""
    return pd.DataFrame(a).ffill(axis=1).to_numpy()

def les_resultat(o):
    """Leser OpenDrift-resultatet fra minnet (o.result, et xarray-datasett)."""
    ds = o.result.transpose("trajectory", "time", ...)
    tider = pd.to_datetime(ds["time"].values)
    n_traj, n_tid = ds["lon"].shape

    def var(navn):
        return ds[navn].values.astype(float) if navn in ds else np.zeros((n_traj, n_tid))

    lon, lat, z = var("lon"), var("lat"), var("z")
    status = var("status")
    gyldig = ~np.isnan(lon)
    sluppet = gyldig.any(axis=1)
    forste = np.where(sluppet, gyldig.argmax(axis=1), n_tid)
    siste = np.where(sluppet, n_tid - 1 - gyldig[:, ::-1].argmax(axis=1), -1)

    meanings = ds["status"].attrs.get("flag_meanings", "").split()
    values = list(np.atleast_1d(ds["status"].attrs.get("flag_values", range(len(meanings)))))
    kode = dict(zip(meanings, values))
    AKTIV, STRANDET = kode.get("active", 0), kode.get("stranded", 1)

    f = {k: ffill(var(k)) for k in ("lon", "lat", "z", "status", "mass_oil", "mass_evaporated", "mass_dispersed")}
    slutt_status = f["status"][np.arange(n_traj), np.maximum(siste, 0)]

    # Deaktivert før siste tidssteg, men siste lagrede status er 'active': slå opp i minnet.
    uklar = sluppet & (siste < n_tid - 1) & (slutt_status == AKTIV)
    if uklar.any():
        try:
            ed = o.elements_deactivated
            id_til_idx = {int(t): i for i, t in enumerate(ds["trajectory"].values)}
            for id_, st in zip(np.atleast_1d(ed.ID), np.atleast_1d(ed.status)):
                i = id_til_idx.get(int(id_))
                if i is not None and uklar[i]:
                    slutt_status[i] = st
        except Exception as e:
            print("  Fant ikke deaktiveringsstatus i minnet:", e)
    deaktivert = sluppet & (siste < n_tid - 1)
    return dict(ds=ds, tider=tider, n_tid=n_tid, f=f, sluppet=sluppet, forste=forste, siste=siste,
                slutt_status=slutt_status, deaktivert=deaktivert, AKTIV=AKTIV, STRANDET=STRANDET)

def flak_polygon(lon, lat):
    if len(lon) == 0:
        return None
    pkt = gpd.GeoSeries(gpd.points_from_xy(lon, lat), crs=CRS_LAGRING).to_crs(CRS_METRISK)
    flak = pkt.buffer(POLYGON_BUFFER_M, quad_segs=8).union_all().buffer(-POLYGON_BUFFER_M / 2)
    if LAND is not None:
        flak = flak.difference(LAND)
    flak = flak.simplify(FORENKLING_M)
    return None if flak.is_empty else flak

def etterbehandle(rad, res):
    f, n_tid = res["f"], res["n_tid"]
    rader, utenfor_maks = [], 0.0
    for k in range(n_tid):
        sluppet_k = res["forste"] <= k
        if not sluppet_k.any():
            continue
        er_deakt_k = res["deaktivert"] & (k > res["siste"])
        strandet_k = (er_deakt_k | (k == res["siste"])) & (res["slutt_status"] == res["STRANDET"]) & sluppet_k
        aktiv_k = sluppet_k & ~er_deakt_k & ~strandet_k
        overflate = aktiv_k & (f["z"][:, k] >= -0.01)
        under = aktiv_k & (f["z"][:, k] < -0.01)

        olje, fordampet, disp = f["mass_oil"][:, k], f["mass_evaporated"][:, k], f["mass_dispersed"][:, k]
        startmasse = olje + fordampet + disp  # partikkelens opprinnelige oljemasse
        i_flak = overflate & (olje >= MIN_MASSEANDEL_FLAK * startmasse)
        totalt = np.nansum((olje + fordampet + disp)[sluppet_k])
        pct = lambda m: round(100 * float(np.nansum(m)) / totalt, 2) if totalt > 0 else 0.0

        # Andel aktive partikler utenfor NorKyst-domenet (får strøm = 0 som fallback)
        if aktiv_k.any():
            x, y = norkyst.lonlat2xy(f["lon"][aktiv_k, k], f["lat"][aktiv_k, k])
            ute = ~((x >= norkyst.xmin) & (x <= norkyst.xmax) & (y >= norkyst.ymin) & (y <= norkyst.ymax))
            utenfor_maks = max(utenfor_maks, 100 * float(np.mean(ute)))

        tid = res["tider"][k]
        rader.append({
            "installation": rad["name"], "oil_type": rad["oil_type"],
            "hour": int(round((tid - pd.Timestamp(STARTTID)).total_seconds() / 3600)),
            "time_utc": tid.strftime("%Y-%m-%dT%H:%M:%SZ"),
            "pct_surface": pct(olje[overflate]), "pct_submerged": pct(olje[under]),
            "pct_evaporated": pct(fordampet[sluppet_k]), "pct_dispersed": pct(disp[sluppet_k]),
            "pct_stranded": pct(olje[strandet_k]),
            "n_surface": int(overflate.sum()), "n_slick": int(i_flak.sum()),
            "geometry": flak_polygon(f["lon"][i_flak, k], f["lat"][i_flak, k]),
        })

    flak = gpd.GeoDataFrame(rader, geometry="geometry", crs=CRS_METRISK)
    flak["area_km2"] = (flak.geometry.area / 1e6).round(2).fillna(0.0)

    s = res["sluppet"] & (res["slutt_status"] == res["STRANDET"])
    idx = np.maximum(res["siste"][s], 0)
    strandet = gpd.GeoDataFrame({
        "installation": rad["name"], "oil_type": rad["oil_type"],
        "time_utc": [res["tider"][i].strftime("%Y-%m-%dT%H:%M:%SZ") for i in idx],
        "hour": [int(round((res["tider"][i] - pd.Timestamp(STARTTID)).total_seconds() / 3600)) for i in idx],
        "mass_kg": np.round(f["mass_oil"][s, :][np.arange(s.sum()), idx], 1),
    }, geometry=gpd.points_from_xy(f["lon"][s, :][np.arange(s.sum()), idx],
                                   f["lat"][s, :][np.arange(s.sum()), idx]), crs=CRS_LAGRING)
    return flak, strandet, utenfor_maks

def skriv_geojson(gdf, sti):
    ut = gdf.to_crs(CRS_LAGRING)
    ut["geometry"] = shapely.set_precision(ut.geometry.values, 1e-5)
    sti.write_text(ut.to_json(drop_id=True, na="null"), encoding="utf-8")

## 8. Kjør én simulering per innretning
Hver kjøring lagrer:
- `spills/<slug>.geojson`: ett polygon per time
- `spills/<slug>_stranded.geojson`: strandede partikler som punkter
- `plots/<slug>_kart.png`: OpenDrifts kartplott
- `plots/<slug>_oljebudsjett.png`: massebalansen over tid

In [ ]:
import time
import matplotlib.pyplot as plt

oppsummering = []
for _, rad in innretninger.iterrows():
    t0 = time.time()
    print(f"\n=== {rad['name']} ({rad['oil_type']}) ===")
    o = lag_modell(rad)
    vis_oppsett(o)
    o.run(duration=timedelta(hours=VARIGHET), time_step=timedelta(minutes=TIDSSTEG_MIN),
          time_step_output=timedelta(hours=OUTPUT))

    for navn, fn in (("kart", lambda p: o.plot(filename=p)),
                     ("oljebudsjett", lambda p: o.plot_oil_budget(filename=p))):
        try:
            fn(str(UT_DIR / "plots" / f"{rad['slug']}_{navn}.png"))
        except Exception as e:
            print(f"  Plott '{navn}' feilet: {e}")
        plt.close("all")

    res = les_resultat(o)
    flak, strandet, utenfor = etterbehandle(rad, res)
    skriv_geojson(flak, UT_DIR / "spills" / f"{rad['slug']}.geojson")
    skriv_geojson(strandet, UT_DIR / "spills" / f"{rad['slug']}_stranded.geojson")

    siste = flak.iloc[-1]
    sum_pct = siste[["pct_surface", "pct_submerged", "pct_evaporated", "pct_dispersed", "pct_stranded"]].sum()
    oppsummering.append({
        "installation": rad["name"], "oil_type": rad["oil_type"], "hours": int(siste["hour"]),
        "area_km2_end": siste["area_km2"], "area_km2_max": flak["area_km2"].max(),
        **{k: siste[k] for k in ("pct_surface", "pct_submerged", "pct_evaporated", "pct_dispersed", "pct_stranded")},
        "pct_sum": round(sum_pct, 1), "n_stranded": len(strandet),
        "max_pct_outside_norkyst": round(utenfor, 1), "seconds": round(time.time() - t0),
    })
    print(f"  Ferdig på {time.time() - t0:.0f} s. Strandet: {len(strandet)} partikler. "
          f"Maks {utenfor:.1f} % av aktive partikler utenfor NorKyst. Sum massebalanse: {sum_pct:.1f} %")
    if utenfor > 0:
        print("  ADVARSEL: noen partikler har vært utenfor NorKyst800 og har da bare drevet med vind/bølger.")

oppsummering = pd.DataFrame(oppsummering)
oppsummering.to_csv(UT_DIR / "oppsummering.csv", index=False)
display(oppsummering)

## 9. Pakk og last ned
Zip-fila inneholder `spills/`, `plots/` og `oppsummering.csv`. Etter full kjøring:
1. Pakk `spills/` ut til `data/processed/spills/` i repoet.
2. Legg plottene der rapporten trenger dem.
3. Commit og push.

Resultater fra testmodus (`_TEST`) skal ikke committes.

In [ ]:
import shutil
zip_sti = shutil.make_archive(f"ingunn_resultater{SUFFIKS}", "zip", root_dir=UT_DIR)
print("Laget", zip_sti)
try:
    from google.colab import files
    files.download(zip_sti)
except ImportError:
    print("Ikke i Colab – hent fila fra arbeidsmappen.")